# Day 2 上午：预训练与 SFT（~3h）

## 学习目标

1. 体验从零预训练（Pretraining）的完整流程
2. 理解 Base Model → Chat Model 的完整链路
3. 完成 SFT（Supervised Fine-Tuning）全流程实操
4. 知道什么场景需要全量微调

### 时间安排

| 时间 | 内容 | 类型 |
|:---|:---|:---|
| 0:00-0:15 | LLM 生命周期：pretrain → SFT → align → deploy | 讲解 |
| 0:15-0:45 | 预训练 demo：在中文语料上训练 TinyLlama | 代码随讲 |
| 0:45-0:55 | 练习1：修改训练数据，观察输出质量变化 | 练习 |
| 0:55-1:10 | 练习2：分析 loss 曲线，诊断过拟合 | 练习 |
| 1:10-1:20 | 休息 | |
| 1:20-1:35 | SFT 动机：Base model 不会聊天 | 演示 |
| 1:35-1:50 | ChatML + Loss Masking | 代码随讲 |
| 1:50-2:00 | 练习3：实现 SFT loss masking | 练习 |
| 2:00-2:25 | SFT 训练实战 | 代码随讲 |
| 2:25-2:35 | 练习4：对比 base vs SFT 准确率 | 练习 |
| 2:35-2:50 | 练习5：调超参（lr, epochs）观察效果 | 练习 |
| 2:50-3:00 | Mini-project：展示 SFT 前后对比 | 演示 |

In [ ]:
# 关闭 HuggingFace 进度条和无关警告
import os, warnings
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore", message=".*tie_word_embeddings.*")
warnings.filterwarnings("ignore", message=".*UNEXPECTED.*")

import sys
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), "utils"))
sys.path.insert(0, "utils")


In [ ]:
# ── 中文字体 + 图表美化 ──
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager as fm

font_path = "../fonts/NotoSansCJKsc-Regular.otf"
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)

plt.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

---
## Part 1：LLM 生命周期全景（0:00-0:15）

一个大语言模型从"出生"到"上岗"，要经过四个阶段：

| 阶段 | 做什么 | 数据 | 产出 | 企业通常从哪进入 |
|:---|:---|:---|:---|:---|
| **预训练（Pretraining）** | 在海量文本上学习语言规律 | TB 级无标注文本 | Base Model（只会续写） | 很少自己做 |
| **SFT（指令微调）** | 教模型遵循指令、按格式回答 | 万~十万条 instruction-response | Chat Model（能对话） | **最常见切入点** |
| **对齐（Alignment）** | 让输出更安全、更符合人类偏好 | 偏好对比数据 | Aligned Model | 高阶需求 |
| **部署（Deployment）** | 量化、缓存、API 化 | 无 | Production Service | 必须做 |

### 关键认知

- **预训练**给模型"知识"和"语言能力"，但它不知道怎么跟人对话
- **SFT**教模型"行为模式"——用 instruction-response 数据把能力激活为对话能力
- 企业落地最常见路径：拿开源 Base Model → SFT 适配业务场景
- 今天上午我们按正确顺序体验：**先预训练，再 SFT**

---
## Part 2：预训练 Demo（0:15-0:45）

> 从零训练一个能生成中文文本的小型语言模型。
>
> 体验完整流程：**数据准备 → 词表构建 → 模型定义 → 训练循环 → 文本生成**

### 2.1 环境与依赖

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import time
import math

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

### 2.2 准备数据：中文预训练语料

预训练的核心：给模型大量文本，让它学习"给定前文，预测下一个字"。

In [ ]:
# 读取本地中文预训练语料
from pathlib import Path
from collections import Counter

def resolve_data_dir():
    candidates = [Path.cwd(), Path.cwd().parent]
    for base in candidates:
        data_dir = base / "data"
        if data_dir.exists():
            return str(data_dir)
    return os.path.join(os.getcwd(), "data")

DATA_DIR = resolve_data_dir()
PROJECT_ROOT = str(Path(DATA_DIR).parent)
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)
data_path = os.path.join(DATA_DIR, "custom_pretrain_corpus.txt")
BEST_MODEL_PATH = os.path.join(MODELS_DIR, "ch7_pretrain_best_model.pt")

if not os.path.exists(data_path):
    raise FileNotFoundError(f"未找到预训练语料: {data_path}")

with open(data_path, "r", encoding="utf-8") as f:
    text = f.read()

# 基本统计
num_lines = text.count('\n')
cjk_chars = [c for c in text if '\u4e00' <= c <= '\u9fff']

print(f"语料统计:")
print(f"  总字符数:   {len(text):>10,}")
print(f"  总行数:     {num_lines:>10,}")
print(f"  中文字符数: {len(cjk_chars):>10,}  ({len(cjk_chars)/len(text)*100:.1f}%)")
print(f"\n前300字符预览:")
print(text[:300])

In [ ]:
# 构建字符级词表（Character-level Vocabulary）
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

fallback_id = stoi.get(' ', stoi.get('\n', 0))
encode = lambda s: [stoi.get(c, fallback_id) for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"vocab_size: {vocab_size}")
test_str = text[:10]
encoded = encode(test_str)
decoded = decode(encoded)
print(f"\nEncode/Decode 测试:")
print(f"  原文:   {test_str}")
print(f"  编码:   {encoded}")
print(f"  解码:   {decoded}")

### 2.3 创建 Dataset 和 DataLoader

预训练用**滑动窗口**从语料中构造 (input, target) 对：

```
原始文本: "深度学习是用多层..."    block_size=5

样本1: input=[深,度,学,习,是]  target=[度,学,习,是,用]
样本2: input=[度,学,习,是,用]  target=[学,习,是,用,多]
```

target 就是 input 右移一位——这就是 Next-Token Prediction。

In [ ]:
class TextDataset(Dataset):
    """文本数据集：每个样本是 (input, target) 对，target 是 input 右移一位"""
    def __init__(self, text, block_size):
        self.data = torch.tensor(encode(text), dtype=torch.long)
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.block_size]
        y = self.data[idx + 1:idx + self.block_size + 1]
        return x, y

# 训练/验证集划分（90% / 10%）
n = int(0.9 * len(text))
train_text = text[:n]
val_text = text[n:]

block_size = 64   # 上下文长度
batch_size = 32

train_dataset = TextDataset(train_text, block_size)
val_dataset = TextDataset(val_text, block_size)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"训练集: {len(train_dataset):,} 个样本")
print(f"验证集: {len(val_dataset):,} 个样本")
print(f"每 batch: {batch_size} x {block_size} = {batch_size * block_size:,} tokens")

# 查看一个样本
x, y = next(iter(train_loader))
print(f"\nBatch shape: x={x.shape}, y={y.shape}")
print(f"输入: {decode(x[0].tolist())}")
print(f"目标: {decode(y[0].tolist())}")

### 2.4 定义模型：一个小型 GPT

我们用 4 层 Transformer（约 50 万参数）来演示预训练流程。

核心组件：Token Embedding + Position Embedding + Self-Attention + MLP

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head = config['n_head']
        self.n_embd = config['n_embd']
        self.head_dim = config['n_embd'] // config['n_head']
        self.c_attn = nn.Linear(config['n_embd'], 3 * config['n_embd'])
        self.c_proj = nn.Linear(config['n_embd'], config['n_embd'])
        self.dropout = nn.Dropout(config['dropout'])
        self.register_buffer("mask", torch.tril(
            torch.ones(config['block_size'], config['block_size'])
        ).view(1, 1, config['block_size'], config['block_size']))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / np.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config['n_embd'], 4 * config['n_embd'])
        self.c_proj = nn.Linear(4 * config['n_embd'], config['n_embd'])
        self.dropout = nn.Dropout(config['dropout'])

    def forward(self, x):
        return self.dropout(self.c_proj(F.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config['n_embd'])
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config['n_embd'])
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config['vocab_size'], config['n_embd']),
            wpe=nn.Embedding(config['block_size'], config['n_embd']),
            drop=nn.Dropout(config['dropout']),
            h=nn.ModuleList([Block(config) for _ in range(config['n_layer'])]),
            ln_f=nn.LayerNorm(config['n_embd']),
        ))
        self.lm_head = nn.Linear(config['n_embd'], config['vocab_size'], bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config['block_size'] else idx[:, -self.config['block_size']:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


# 模型配置
config = {
    'vocab_size': vocab_size,
    'block_size': block_size,
    'n_layer': 4,
    'n_head': 4,
    'n_embd': 128,
    'dropout': 0.1,
}

model = GPT(config).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {total_params:,}")
print(f"模型大小约: {total_params * 4 / 1024 / 1024:.2f} MB (FP32)")

In [ ]:
# 训练前：保存随机初始化模型的生成结果，训练后对比
expected_init_loss = math.log(vocab_size)
print(f"理论初始 loss = ln(vocab_size) = ln({vocab_size}) = {expected_init_loss:.4f}")
print(f"（随机初始化时每个 token 概率 = 1/{vocab_size}，-log(1/{vocab_size}) = {expected_init_loss:.2f}）\n")

model.eval()
pretrain_prompts = [line + "\n" for line in text.splitlines()[:3]]
pretrain_before = []
for prompt in pretrain_prompts:
    ctx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    gen = model.generate(ctx, max_new_tokens=80, temperature=0.8, top_k=40)
    pretrain_before.append(decode(gen[0].tolist()))

print("训练前（随机初始化）模型生成:")
for i, (p, g) in enumerate(zip(pretrain_prompts, pretrain_before)):
    print(f"\n[Prompt {i+1}] {p.strip()}")
    print(f"[Output]  {g[:120]}...")
model.train()

### 2.5 训练循环

预训练的损失函数是 **Cross-Entropy Loss（交叉熵）**：模型对每个位置预测下一个 token 的概率分布，与真实 token 算交叉熵。

$$\mathcal{L} = -\frac{1}{T}\sum_{t=1}^{T} \log P(x_t | x_1, ..., x_{t-1}; \theta)$$

In [ ]:
# 训练配置
learning_rate = 3e-4
max_iters = 500          # 降低迭代次数以节省时间（原2000次）
# 💡 如果使用CPU且仍然较慢，可进一步降至200次，并减小batch_size
eval_interval = 200
eval_iters = 50

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# 简易学习率调度：线性 warmup + cosine decay
warmup_steps = max(1, int(0.1 * max_iters))

def get_lr(step):
    if step < warmup_steps:
        return learning_rate * step / warmup_steps
    progress = (step - warmup_steps) / max(1, max_iters - warmup_steps)
    return learning_rate * 0.1 + 0.5 * (learning_rate - learning_rate * 0.1) * (1 + math.cos(math.pi * progress))

@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {'train': 0, 'val': 0}
    for split, loader in [('train', train_loader), ('val', val_loader)]:
        total_loss, count = 0, 0
        for x, y in loader:
            if count >= eval_iters:
                break
            x, y = x.to(device), y.to(device)
            _, loss = model(x, y)
            total_loss += loss.item()
            count += 1
        losses[split] = total_loss / count
    model.train()
    return losses

print(f"训练配置:")
print(f"  最大迭代: {max_iters:,}")
print(f"  每次迭代: {batch_size * block_size:,} tokens")
print(f"  Warmup:   {warmup_steps} 步")
print(f"  学习率:   {learning_rate}")
print(f"\n  (CPU 上预计 2-5 分钟)")
print("=" * 60)
print("开始训练...")

In [ ]:
# 训练主循环
train_losses = []
val_losses = []
best_val_loss = float('inf')

start_time = time.time()
iter_count = 0
tokens_processed = 0
tokens_per_iter = batch_size * block_size

for epoch in range(max_iters // len(train_loader) + 1):
    for x, y in train_loader:
        if iter_count >= max_iters:
            break

        # 更新学习率
        lr = get_lr(iter_count)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        x, y = x.to(device), y.to(device)
        logits, loss = model(x, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        tokens_processed += tokens_per_iter

        # 定期评估
        if iter_count % eval_interval == 0:
            losses = estimate_loss()
            train_losses.append(losses['train'])
            val_losses.append(losses['val'])

            elapsed = time.time() - start_time
            ppl_val = math.exp(min(losses['val'], 20))

            print(f"Iter {iter_count:4d} | "
                  f"Train: {losses['train']:.4f} | "
                  f"Val: {losses['val']:.4f} (ppl={ppl_val:.1f}) | "
                  f"{tokens_processed / elapsed:,.0f} tok/s | "
                  f"{elapsed:.1f}s")

            if losses['val'] < best_val_loss:
                best_val_loss = losses['val']
                torch.save(model.state_dict(), BEST_MODEL_PATH)

        iter_count += 1

total_time = time.time() - start_time
print("=" * 60)
print(f"训练完成! 最佳 Val Loss: {best_val_loss:.4f} (ppl={math.exp(min(best_val_loss,20)):.1f})")
print(f"总耗时: {total_time:.1f}s")

In [ ]:
# 可视化训练过程
fig, ax1 = plt.subplots(figsize=(10, 5))

eval_steps = list(range(len(train_losses)))
ax1.plot(eval_steps, train_losses, label='Train Loss', color='#3B82F6', linewidth=2)
ax1.plot(eval_steps, val_losses, label='Val Loss', color='#F97316', linewidth=2)

expected_init = math.log(vocab_size)
ax1.axhline(y=expected_init, color='#EF4444', linestyle='--', alpha=0.6,
            label=f'ln(vocab_size) = {expected_init:.2f}')

ax1.set_xlabel('评估步骤')
ax1.set_ylabel('Loss')
ax1.set_title('预训练过程：Loss 曲线')
ax1.legend()
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"最终 Train Loss: {train_losses[-1]:.4f}")
print(f"最终 Val Loss:   {val_losses[-1]:.4f}")

In [ ]:
# 加载最佳模型并生成文本
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

prompt = text.splitlines()[0] + "\n"
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

max_new = 200
gen_start = time.time()
generated = model.generate(context, max_new_tokens=max_new, temperature=0.8, top_k=40)
gen_time = time.time() - gen_start

full_text = decode(generated[0].tolist())
prompt_len = len(prompt)

print(f"{'=' * 60}")
print(f"  Prompt: {prompt.strip()}")
print(f"{'─' * 60}")
print(f"  Generated ({max_new} tokens):")
print(f"  {full_text[prompt_len:]}")
print(f"{'=' * 60}")
print(f"  生成速度: {max_new / gen_time:.1f} tokens/sec")

In [ ]:
# 训练前后生成对比
print("=" * 70)
print("训练前 vs 训练后 生成文本对比")
print("=" * 70)

model.eval()
for i, prompt in enumerate(pretrain_prompts):
    ctx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    gen = model.generate(ctx, max_new_tokens=80, temperature=0.8, top_k=40)
    after = decode(gen[0].tolist())

    print(f"\n[Prompt {i+1}] {prompt.strip()}")
    print(f"  训练前: {pretrain_before[i][len(prompt):80+len(prompt)]}")
    print(f"  训练后: {after[len(prompt):80+len(prompt)]}")

---
## 练习 1：修改训练数据，观察输出质量变化（0:45-0:55）⭐ 入门

**目标**：理解预训练数据质量对模型的直接影响。

**任务**：
1. 创建一份"低质量"语料（重复、噪声文本）
2. 在低质量语料上快速训练 500 步
3. 对比两个模型的生成质量

**思考**：为什么大模型公司都在拼数据质量？

**预期输出**：
```
Prompt: <第一行文本>
好数据训练: <连贯的中文文本>
坏数据训练: <重复/乱码文本>
结论: 数据质量直接决定模型质量!
```

In [ ]:
# ====== 练习 1：修改训练数据，观察输出质量变化 ======

# 步骤 1：构造低质量数据（大量重复 + 无意义文本）
# ↓↓↓ 你的代码 ↓↓↓
noisy_text = "这是重复文本。\n" * 300 + "啊" * 200
# ↑↑↑ 你的代码 ↑↑↑

# 步骤 2：用同样的词表构建数据集
# ↓↓↓ 你的代码 ↓↓↓
noisy_dataset = TextDataset(noisy_text, block_size)
noisy_loader = DataLoader(noisy_dataset, batch_size=batch_size, shuffle=True)
# ↑↑↑ 你的代码 ↑↑↑

# 步骤 3：训练新模型 500 步
noisy_model = GPT(config).to(device)
noisy_opt = torch.optim.AdamW(noisy_model.parameters(), lr=3e-4)

noisy_iters = 0
for epoch in range(500 // max(1, len(noisy_loader)) + 1):
    for x, y in noisy_loader:
        if noisy_iters >= 500:
            break
        x, y = x.to(device), y.to(device)
        # ↓↓↓ 你的代码（前向传播、反向传播、参数更新） ↓↓↓
        logits, loss = noisy_model(x, y)
        noisy_opt.zero_grad()
        loss.backward()
        noisy_opt.step()
        # ↑↑↑ 你的代码 ↑↑↑
        noisy_iters += 1

# --------- 验证（不要修改） ---------
test_prompt = text.splitlines()[0] + "\n"
ctx = torch.tensor([encode(test_prompt)], dtype=torch.long, device=device)

model.eval()
noisy_model.eval()

good_gen = decode(model.generate(ctx, max_new_tokens=80, temperature=0.8, top_k=40)[0].tolist())
noisy_gen = decode(noisy_model.generate(ctx, max_new_tokens=80, temperature=0.8, top_k=40)[0].tolist())

print(f"Prompt: {test_prompt.strip()}")
print(f"\n好数据训练: {good_gen[len(test_prompt):][:100]}")
print(f"\n坏数据训练: {noisy_gen[len(test_prompt):][:100]}")
print(f"\n结论: 数据质量直接决定模型质量!")

assert noisy_iters == 500, f"训练步数应为500，实际为{noisy_iters}"
print("\n✅ 练习 1 完成！")

<details><summary>提示 1：构造低质量数据</summary>

```python
noisy_text = "这是重复文本。\n" * 300 + "啊" * 200
```
</details>

<details><summary>提示 2：构建数据集</summary>

```python
noisy_dataset = TextDataset(noisy_text, block_size)
noisy_loader = DataLoader(noisy_dataset, batch_size=batch_size, shuffle=True)
```
</details>

<details><summary>提示 3：训练循环</summary>

```python
logits, loss = noisy_model(x, y)
noisy_opt.zero_grad()
loss.backward()
noisy_opt.step()
```
</details>

---
## 练习 2：分析 Loss 曲线，诊断过拟合（0:55-1:10）⭐ 入门

**目标**：学会从 loss 曲线判断模型训练状态。

**任务**：
1. 用一个极小数据集（仅 10 行文本）训练模型
2. 画出 train loss 和 val loss 曲线
3. 判断：是过拟合（overfitting）还是欠拟合（underfitting）？

**判断标准**：
- 过拟合：train loss 持续下降，val loss 先降后升
- 欠拟合：train loss 和 val loss 都很高，模型没学到东西

**预期输出**：
```
一张折线图，train loss 持续下降，val loss 先降后升
分析结论: 典型的过拟合模式
```

In [ ]:
# ====== 练习 2：分析 Loss 曲线，诊断过拟合 ======

# 用极小数据集演示过拟合
tiny_text = "\n".join(text.splitlines()[:10])
tiny_val_text = "\n".join(text.splitlines()[10:20])

tiny_train_ds = TextDataset(tiny_text, block_size)
tiny_val_ds = TextDataset(tiny_val_text, block_size)
tiny_train_loader = DataLoader(tiny_train_ds, batch_size=8, shuffle=True)
tiny_val_loader = DataLoader(tiny_val_ds, batch_size=8, shuffle=False)

tiny_model = GPT(config).to(device)
tiny_opt = torch.optim.AdamW(tiny_model.parameters(), lr=3e-4)

# 训练 1000 步，每 50 步记录 train/val loss
tiny_train_losses = []
tiny_val_losses = []
tiny_iter = 0

for epoch in range(1000 // max(1, len(tiny_train_loader)) + 1):
    for x, y in tiny_train_loader:
        if tiny_iter >= 1000:
            break
        x, y = x.to(device), y.to(device)

        # ↓↓↓ 你的代码（训练一步） ↓↓↓
        logits, loss = tiny_model(x, y)
        tiny_opt.zero_grad()
        loss.backward()
        tiny_opt.step()
        # ↑↑↑ 你的代码 ↑↑↑

        # 每 50 步记录 loss（不要修改）
        if tiny_iter % 50 == 0:
            tiny_model.eval()
            t_loss, v_loss = 0, 0
            with torch.no_grad():
                for tx, ty in tiny_train_loader:
                    tx, ty = tx.to(device), ty.to(device)
                    _, tl = tiny_model(tx, ty)
                    t_loss = tl.item()
                    break
                for vx, vy in tiny_val_loader:
                    vx, vy = vx.to(device), vy.to(device)
                    _, vl = tiny_model(vx, vy)
                    v_loss = vl.item()
                    break
            tiny_train_losses.append(t_loss)
            tiny_val_losses.append(v_loss)
            tiny_model.train()

        tiny_iter += 1

# --------- 可视化（不要修改） ---------
plt.figure(figsize=(8, 4))
plt.plot(tiny_train_losses, label='Train Loss', color='#3B82F6')
plt.plot(tiny_val_losses, label='Val Loss', color='#F97316')
plt.xlabel('记录次数（每 50 步）')
plt.ylabel('Loss')
plt.title('过拟合演示：Train vs Val Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# --------- 验证 ---------
assert len(tiny_train_losses) > 0, "没有记录到训练 loss"
assert len(tiny_val_losses) > 0, "没有记录到验证 loss"
print("\n✅ 练习 2 完成！")
print("分析结论: 典型的过拟合模式")
print("- Train loss 持续下降（模型在记住训练数据）")
print("- Val loss 先降后升（在验证集上表现变差）")
print("原因：数据集太小（仅 10 行），模型参数远超过数据量")

<details><summary>提示：训练一步的代码</summary>

```python
logits, loss = tiny_model(x, y)
tiny_opt.zero_grad()
loss.backward()
tiny_opt.step()
```
</details>

---
*休息 10 分钟（1:10-1:20）*

---

---
### 🌉 从玩具模型到真实模型

到这里，我们已经用一个小型模型走通了预训练的完整流程：数据准备 → 模型定义 → 训练循环 → 文本生成。

接下来，我们将切换到一个**真正在大规模中文语料上预训练过的模型**（`uer/gpt2-chinese-cluecorpussmall`）。你会发现：
- 代码结构几乎相同——HuggingFace 的 `transformers` 库帮我们封装好了
- 但模型的能力天差地别——这就是"大数据 + 大模型"的威力
- 我们还将在这个真实模型上进行 **SFT（监督微调）**，让它学会按指令回答问题

---

## Part 3：SFT 指令微调（1:20-3:00）

> 预训练给了模型"知识"，但它只会续写文本，不会对话。
>
> SFT（Supervised Fine-Tuning）教模型"行为"——用 instruction-response 数据把能力激活为对话能力。

### 3.1 SFT 动机：Base Model 不会聊天（1:20-1:35）

| | Base Model（预训练后） | Chat Model（SFT 后） |
|:---|:---|:---|
| 训练目标 | 预测下一个 token | 学习对话格式 |
| 输入 | 连续文本 | instruction-response 对 |
| 行为 | 续写文本 | 按指令回答 |
| 比喻 | 读了所有书的学生 | 经过培训的客服 |

In [ ]:
# ── SFT 部分：使用 HuggingFace Transformers ──
import json
import random
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

USE_GPU = device == 'cuda'

# 路径设置
DATA_DIR_P = Path(DATA_DIR)
MODEL_DIR = Path(MODELS_DIR)
DATA_DIR_P.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"data dir: {DATA_DIR_P}")
print(f"model dir: {MODEL_DIR}")

In [ ]:
# 加载中文 GPT-2 Base Model
import warnings, logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

model_name = "uer/gpt2-chinese-cluecorpussmall"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    base_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    sft_model  = AutoModelForCausalLM.from_pretrained(model_name).to(device)

if tokenizer.pad_token_id is None:
    if tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})
        base_model.resize_token_embeddings(len(tokenizer))
        sft_model.resize_token_embeddings(len(tokenizer))

for m in (base_model, sft_model):
    m.config.pad_token_id = tokenizer.pad_token_id
    m.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"✅ 模型加载完成: {model_name}")
print(f"   词表大小: {len(tokenizer)}, pad_token_id: {tokenizer.pad_token_id}")
print(f"   参数量: {sum(p.numel() for p in base_model.parameters()):,}")


In [ ]:
# 演示：Base Model 不会对话
# 给它一个客服场景的指令，看它怎么回答

demo_instruction = "请读取用户反馈并判断主要诉求类别。\n用户反馈：订单OD20240315-1234，下单三天了仍显示待发货，请帮我尽快处理。\n请严格输出两行：\n类别：<一个类别>\n理由：<一句话>"
demo_prompt = f"{demo_instruction}\n\n助手："

inp = tokenizer(demo_prompt, return_tensors="pt", add_special_tokens=False).to(device)

base_model.eval()
with torch.no_grad():
    out = base_model.generate(
        **inp,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

gen_text = tokenizer.decode(out[0][inp['input_ids'].shape[-1]:], skip_special_tokens=True)
print("指令：")
print(demo_instruction[:80] + "...")
print(f"\nBase Model 的回答：")
print(gen_text.replace(' ', '')[:200])
print("\n>> Base Model 不知道要按格式回答，只是在续写文本！")
print(">> 这就是为什么我们需要 SFT。")

### 3.2 ChatML 格式 + Loss Masking（1:35-1:50）

SFT 有两个核心概念，我们**一次讲清楚**：

#### ChatML 格式

ChatML（Chat Markup Language）是对话数据的标准格式，让模型区分不同角色：

```
<|im_start|>user
今天天气怎么样？<|im_end|>
<|im_start|>assistant
今天天气晴朗，适合出门。<|im_end|>
```

#### Loss Masking：只训练 assistant 的回复

关键思想：我们**只让模型学习生成 assistant 的回复部分**，不学习 user 的提问。

```
input_ids:  [用户问题的tokens...] [助手回复的tokens...] [EOS]
labels:     [-100 -100 -100 ... ] [助手回复的tokens...] [EOS]
             ^^^^^^^^^^^^^^^^^^^   ^^^^^^^^^^^^^^^^^^^^^^^^
             不计算 loss            计算 loss
```

`-100` 是 PyTorch CrossEntropyLoss 的默认忽略值，这些位置不参与梯度计算。

In [ ]:
# ChatML 格式演示
from typing import Dict, List

def format_conversation(messages: List[Dict[str, str]]) -> str:
    """将对话转为 ChatML 格式"""
    formatted = ""
    for msg in messages:
        formatted += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
    return formatted

example_conv = [
    {"role": "user", "content": "什么是机器学习？"},
    {"role": "assistant", "content": "机器学习是让模型从数据中学习规律并做预测的方法。"},
]

print("ChatML 格式示例:")
print("=" * 50)
print(format_conversation(example_conv))

In [ ]:
# Loss Masking 可视化
# 简化版演示：看 input_ids 和 labels 的对应关系

demo_input = "用户：天气如何？\n\n助手：今天晴朗。"
demo_ids = tokenizer(demo_input, add_special_tokens=False)["input_ids"]
demo_tokens = tokenizer.convert_ids_to_tokens(demo_ids)

# 假设 "助手：" 之后的内容是 assistant 部分
# 找到分界点
assistant_marker = tokenizer("助手：", add_special_tokens=False)["input_ids"]
split_idx = len(demo_ids)  # default
for i in range(len(demo_ids) - len(assistant_marker) + 1):
    if demo_ids[i:i+len(assistant_marker)] == assistant_marker:
        split_idx = i + len(assistant_marker)
        break

# 构造 labels
demo_labels = [-100] * split_idx + demo_ids[split_idx:]

print("Loss Masking 示例:")
print("=" * 60)
print(f"{'位置':<5} {'Token':<10} {'input_id':<10} {'label':<10} {'状态'}")
print("-" * 60)
for i, (tok, iid) in enumerate(zip(demo_tokens, demo_ids)):
    lb = demo_labels[i] if i < len(demo_labels) else -100
    status = "MASK (不算loss)" if lb == -100 else "TRAIN (算loss)"
    print(f"{i:<5} {tok:<10} {iid:<10} {lb:<10} {status}")

---
## 练习 3：实现 SFT Loss Masking（1:50-2:00）⭐ 入门

**目标**：亲手实现 Loss Masking 的核心逻辑。

**规则**：
- `assistant_start_idx` 之前的位置 → `labels = -100`（不计算 loss）
- `assistant_start_idx` 及之后的位置 → `labels = 原 token_id`（计算 loss）

**预期输出**：
```
正确! 你理解了 SFT Loss Masking 的核心逻辑
  input_ids: [101, 202, 303, 404, 505, 606, 707]
  labels:    [-100, -100, -100, 404, 505, 606, 707]
```

In [ ]:
# ====== 练习 3：实现 SFT Loss Masking ======

def create_sft_labels(input_ids, assistant_start_idx):
    """
    创建 SFT 训练的 labels（简化版）

    参数:
        input_ids: token id 列表，如 [101, 202, 303, 404, 505]
        assistant_start_idx: assistant 回复开始的位置索引

    返回:
        labels: 与 input_ids 等长的列表
                位置 < assistant_start_idx -> -100（不计算 loss）
                位置 >= assistant_start_idx -> 原 token id（计算 loss）
    """
    # ↓↓↓ 你的代码 ↓↓↓
    labels = [-100] * assistant_start_idx + input_ids[assistant_start_idx:]
    # ↑↑↑ 你的代码 ↑↑↑
    return labels


# --------- 验证（不要修改） ---------
test_ids = [101, 202, 303, 404, 505, 606, 707]
assistant_start = 3

your_labels = create_sft_labels(test_ids, assistant_start)
expected_labels = [-100, -100, -100, 404, 505, 606, 707]

assert your_labels == expected_labels, f"结果不正确！你的结果: {your_labels}，期望: {expected_labels}"

print("正确! 你理解了 SFT Loss Masking 的核心逻辑")
print(f"  input_ids: {test_ids}")
print(f"  labels:    {your_labels}")
print(f"  解释：前 {assistant_start} 个 token 是 user 部分（-100），后面是 assistant 部分")
print("\n✅ 练习 3 完成！")

<details><summary>提示 1：列表推导写法</summary>

```python
labels = [-100 if i < assistant_start_idx else input_ids[i] for i in range(len(input_ids))]
```
</details>

<details><summary>提示 2：拼接写法</summary>

```python
labels = [-100] * assistant_start_idx + input_ids[assistant_start_idx:]
```
</details>

### 3.3 SFT 训练实战（2:00-2:25）

我们用电商客服诉求分类任务来完成 SFT 全流程。

**任务**：给定用户反馈文本，模型输出「类别」和「理由」。

In [ ]:
# 构造训练数据
LABELS = ["延迟发货", "退款申请", "地址修改", "物流异常", "售后咨询", "发票问题"]

CLUES = {
    LABELS[0]: ["下单三天了仍显示待发货", "店铺一直没有交给快递", "承诺发货时间已过但状态未更新"],
    LABELS[1]: ["这个商品不想要了希望退款", "订单想取消并退回付款", "请帮我走退货退款流程"],
    LABELS[2]: ["收件人电话和街道写错了要更改", "请把配送地址改到新公司地址", "还没发货，需要修改收货城市"],
    LABELS[3]: ["物流轨迹停在中转站好几天不动", "快递状态长时间不更新", "包裹显示异常签收但我没收到"],
    LABELS[4]: ["收到货后发现机器不工作怎么处理", "商品到货有破损想咨询售后", "已经签收了但使用中遇到故障"],
    LABELS[5]: ["发票抬头填错了需要重开", "请开一张公司增值税发票", "想确认电子发票什么时候能开"],
}

REASONS = {
    LABELS[0]: ["关键信息是订单超过承诺时间仍未发货", "用户的核心诉求是催促尽快发货"],
    LABELS[1]: ["用户明确提出取消订单并退回款项", "反馈重点是退货退款需求"],
    LABELS[2]: ["反馈主要指向收货地址信息更改", "用户的目标是修正配送地址再发货"],
    LABELS[3]: ["轨迹停滞或签收状态异常属于物流问题", "诉求集中在运输进度异常而非商品本身"],
    LABELS[4]: ["问题发生在收货之后，属于售后支持范畴", "用户在咨询质量或故障处理的售后方案"],
    LABELS[5]: ["关键诉求是发票开具或发票信息修改", "反馈内容围绕发票类型与开票流程"],
}

TAG_TASK = "请读取用户反馈并判断主要诉求类别。"

def rand_order_id(rng):
    return f"OD{rng.randint(2024, 2025)}{rng.randint(1, 12):02d}{rng.randint(1, 28):02d}-{rng.randint(1000, 9999)}"

def build_record(rng):
    label = rng.choice(LABELS)
    clue = rng.choice(CLUES[label])
    reason = rng.choice(REASONS[label])
    order_id = rand_order_id(rng)
    user_text = f"订单{order_id}，{clue}，请帮我尽快处理。"
    instruction = (
        f"{TAG_TASK}\n"
        f"用户反馈：{user_text}\n"
        f"请严格输出两行：\n"
        f"类别：<一个类别>\n"
        f"理由：<一句话>"
    )
    response = f"类别：{label}\n理由：{reason}"
    return {"instruction": instruction, "response": response, "expected": label}

def build_split(n, seed):
    rng = random.Random(seed)
    return [build_record(rng) for _ in range(n)]

TRAIN_RECORDS = build_split(2400, 42)
VAL_RECORDS = build_split(300, 43)
TEST_RECORDS = build_split(300, 44)

print(f"train/val/test: {len(TRAIN_RECORDS)}/{len(VAL_RECORDS)}/{len(TEST_RECORDS)}")
print(f"\n样例 instruction:\n{TRAIN_RECORDS[0]['instruction']}")
print(f"\n样例 response:\n{TRAIN_RECORDS[0]['response']}")

In [ ]:
# SFT Dataset：实现 Loss Masking
MAX_LENGTH = 160

def make_prompt(instruction: str) -> str:
    return f"{instruction}\n\n助手："


class SFTDataset(Dataset):
    def __init__(self, records, tokenizer, max_length=160):
        self.samples = []
        self.tok = tokenizer
        self.max_length = max_length

        eos_id = tokenizer.eos_token_id
        if eos_id is None:
            eos_id = tokenizer.pad_token_id or 0

        for r in records:
            prompt = make_prompt(r["instruction"])
            answer = r["response"]

            p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
            a_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]

            if len(a_ids) == 0:
                continue
            max_prompt_len = max_length - len(a_ids) - 1
            if max_prompt_len <= 0:
                continue
            if len(p_ids) > max_prompt_len:
                p_ids = p_ids[-max_prompt_len:]

            input_ids = p_ids + a_ids + [eos_id]
            labels = ([-100] * len(p_ids)) + a_ids + [eos_id]

            if len(input_ids) != len(labels):
                continue
            if any(x is None for x in input_ids):
                continue
            if sum(1 for x in labels if x != -100) == 0:
                continue

            input_ids = [int(x) for x in input_ids]
            labels = [(-100 if x == -100 else int(x)) for x in labels]

            self.samples.append({
                "input_ids": input_ids,
                "labels": labels,
                "attention_mask": [1] * len(input_ids),
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    max_len = max(len(x["input_ids"]) for x in batch)
    ids, labels, masks = [], [], []
    for x in batch:
        l = len(x["input_ids"])
        p = max_len - l
        ids.append(x["input_ids"] + [pad_id] * p)
        labels.append(x["labels"] + [-100] * p)
        masks.append(x["attention_mask"] + [0] * p)
    return {
        "input_ids": torch.tensor(ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(masks, dtype=torch.long),
    }


train_ds = SFTDataset(TRAIN_RECORDS, tokenizer, MAX_LENGTH)
val_ds = SFTDataset(VAL_RECORDS, tokenizer, MAX_LENGTH)

print(f"processed train: {len(train_ds)}/{len(TRAIN_RECORDS)}")
print(f"processed val:   {len(val_ds)}/{len(VAL_RECORDS)}")

In [ ]:
# 可视化 Mask 分布
def show_mask(sample, tok, max_tokens=80):
    """高亮显示 mask 分布：[MASK] vs [TRAIN]"""
    ids = sample["input_ids"]
    labels = sample["labels"]
    unk_id = tok.unk_token_id if tok.unk_token_id is not None else 0
    safe_ids = [unk_id if x is None else int(x) for x in ids]
    toks = tok.convert_ids_to_tokens(safe_ids)
    masked = sum(1 for x in labels if x == -100)
    total = len(labels)
    supervised = total - masked
    print(f"[MASK] 不算 loss: {masked}/{total} ({masked/total*100:.1f}%)")
    print(f"[TRAIN] 算 loss:  {supervised}/{total} ({supervised/total*100:.1f}%)")
    n = min(max_tokens, len(toks))
    for i in range(n):
        tag = "MASK " if labels[i] == -100 else "TRAIN"
        print(f"  [{tag}] {toks[i]}", end="")
        if (i + 1) % 8 == 0:
            print()
    print()

show_mask(train_ds[0], tokenizer)

In [ ]:
# DataLoader
BATCH_SIZE = 8 if USE_GPU else 4
train_loader_sft = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader_sft = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"训练 batches/epoch: {len(train_loader_sft)}")
print(f"验证 batches:       {len(val_loader_sft)}")
print(f"batch size:         {BATCH_SIZE}")

In [ ]:
# 评估函数
def generate_answer(model, instruction, max_new_tokens=64):
    model.eval()
    prompt = make_prompt(instruction)
    inp = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
    eos_id = tokenizer.eos_token_id or tokenizer.pad_token_id or 0
    with torch.no_grad():
        out = model.generate(
            **inp, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id or 0, eos_token_id=eos_id,
        )
    gen = out[0][inp["input_ids"].shape[-1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


def normalize_colon(s):
    return s.replace("\uff1a", ":")


def rank_label(model, instruction, labels):
    """用 NLL 排序选出最可能的标签（比生成更稳定）"""
    model.eval()
    prompt = make_prompt(instruction)
    p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    scores = []
    with torch.no_grad():
        for lb in labels:
            ans = f"类别：{lb}\n理由："
            a_ids = tokenizer(ans, add_special_tokens=False)["input_ids"]
            ids = p_ids + a_ids
            lbl = ([-100] * len(p_ids)) + a_ids
            ids_t = torch.tensor([ids], dtype=torch.long, device=device)
            attn_t = torch.ones_like(ids_t)
            lbl_t = torch.tensor([lbl], dtype=torch.long, device=device)
            out = model(input_ids=ids_t, attention_mask=attn_t, labels=lbl_t)
            scores.append((lb, out.loss.item()))
    scores.sort(key=lambda x: x[1])
    return scores[0][0], scores


EVAL_SAMPLES = 120 if USE_GPU else 60


def eval_acc(model, records, labels, max_samples=None):
    if max_samples is None:
        max_samples = EVAL_SAMPLES
    y_true, y_pred = [], []
    n = min(max_samples, len(records))
    hit = 0
    rows = []
    print(f"评估中 (共 {n} 样本) ...")
    for i in range(n):
        r = records[i]
        pred_label, _ = rank_label(model, r["instruction"], labels)
        pred_text = generate_answer(model, r["instruction"])
        y_true.append(r["expected"])
        y_pred.append(pred_label)
        ok = pred_label == r["expected"]
        hit += int(ok)
        if len(rows) < 5:
            rows.append((r["instruction"], r["expected"], pred_label, pred_text))
        if (i + 1) % 10 == 0 or i == n - 1:
            print(f"  {i+1}/{n} ... acc: {hit/(i+1)*100:.1f}%")
    return hit / max(1, n), rows, y_true, y_pred


# 先评估 Base Model
base_acc, base_rows, base_y_true, base_y_pred = eval_acc(base_model, TEST_RECORDS, LABELS)
print(f"\nBase accuracy: {base_acc*100:.2f}%")

In [ ]:
# SFT 训练
LR = 2e-5 if USE_GPU else 5e-5
EPOCHS = 2
MAX_STEPS = 200 if USE_GPU else 100  # 缩短演示时间

opt = torch.optim.AdamW(sft_model.parameters(), lr=LR, weight_decay=0.01)

print(f"训练配置: LR={LR}, EPOCHS={EPOCHS}, MAX_STEPS={MAX_STEPS}, BATCH={BATCH_SIZE}")


def eval_loss_sft(model, loader):
    model.eval()
    vals = []
    with torch.no_grad():
        for b in loader:
            ids = b["input_ids"].to(device)
            m = b["attention_mask"].to(device)
            y = b["labels"].to(device)
            out = model(input_ids=ids, attention_mask=m, labels=y)
            if not (torch.isnan(out.loss) or torch.isinf(out.loss)):
                vals.append(out.loss.item())
    return sum(vals) / max(1, len(vals))


sft_train_losses = []
sft_val_losses = []
step = 0
t_start = time.time()

for ep in range(EPOCHS):
    sft_model.train()
    ep_vals = []

    for b in train_loader_sft:
        ids = b["input_ids"].to(device)
        m = b["attention_mask"].to(device)
        y = b["labels"].to(device)

        out = sft_model(input_ids=ids, attention_mask=m, labels=y)
        loss = out.loss

        if torch.isnan(loss) or torch.isinf(loss):
            opt.zero_grad(set_to_none=True)
            continue

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sft_model.parameters(), 1.0)
        opt.step()

        step += 1
        ep_vals.append(loss.item())

        if step % 20 == 0:
            sft_train_losses.append(loss.item())
            elapsed = time.time() - t_start
            print(f"step {step:4d} | loss={loss.item():.4f} | {elapsed:.0f}s")

        if step >= MAX_STEPS:
            break

    v = eval_loss_sft(sft_model, val_loader_sft)
    sft_val_losses.append(v)
    ep_mean = sum(ep_vals) / max(1, len(ep_vals))
    print(f"epoch {ep+1}/{EPOCHS} | train={ep_mean:.4f} | val={v:.4f}")

    if step >= MAX_STEPS:
        break

sft_total_time = time.time() - t_start
print(f"\nSFT 训练耗时: {sft_total_time:.1f}s ({sft_total_time/60:.1f} min)")

In [ ]:
# 训练 Loss 曲线
import numpy as np

def moving_average(data, window):
    if len(data) < window:
        return data
    cumsum = np.cumsum(np.insert(data, 0, 0))
    return (cumsum[window:] - cumsum[:-window]) / window

plt.figure(figsize=(10, 4))
plt.plot(sft_train_losses, alpha=0.3, color="C0", label="train (raw)")
ma_window = max(3, min(20, len(sft_train_losses) // 10))
if len(sft_train_losses) >= ma_window:
    ma = moving_average(sft_train_losses, ma_window)
    offset = ma_window // 2
    plt.plot(range(offset, offset + len(ma)), ma, color="C0", linewidth=2,
             label=f"train (MA-{ma_window})")
plt.title("SFT Training Loss")
plt.xlabel("log step (every 20 steps)")
plt.ylabel("loss")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

---
## 练习 4：SFT错误分析（15分钟）⭐⭐ 进阶

准确率只是冰山一角——深入分析模型在**哪些类别**上出错。

**任务**：
1. `eval_acc()` 已提供（直接调用获取结果）
2. 实现 `analyze_errors(predictions, labels)` 按类别统计错误
3. 找出最常见的混淆对（模型最容易把A类误判为B类）
4. 对比 base vs SFT 的错误模式差异
5. 写结论："SFT最大改善了哪类错误？为什么？"

**预期输出**：
```
类别错误统计:
| 类别     | Base正确率 | SFT正确率 | 提升 |
|---------|----------|----------|------|
| 产品质量  | xx%      | xx%      | +xx% |
| 物流配送  | xx%      | xx%      | +xx% |
| ...     |          |          |      |

最常见混淆对: 产品质量 → 售后服务 (xx次)
```

In [ ]:
# ====== 练习 4：SFT 错误分析 ======

from collections import defaultdict, Counter

# 步骤 1：获取 base 和 SFT 模型的预测结果
print("评估 Base 模型和 SFT 模型...")
base_results = []
sft_results = []

for row in TEST_RECORDS[:50]:
    instruction = row["instruction"]
    true_label = row["expected"]

    base_label, _ = rank_label(base_model, instruction, LABELS)
    base_results.append({"true": true_label, "pred": base_label})

    sft_label, _ = rank_label(sft_model, instruction, LABELS)
    sft_results.append({"true": true_label, "pred": sft_label})

print(f"Base 准确率: {sum(1 for r in base_results if r['true']==r['pred'])/len(base_results)*100:.1f}%")
print(f"SFT 准确率: {sum(1 for r in sft_results if r['true']==r['pred'])/len(sft_results)*100:.1f}%")

# 步骤 2：实现错误分析函数
def analyze_errors(results, all_labels):
    """
    按类别统计准确率，找出最常见的混淆对

    参数:
        results: [{"true": label, "pred": label}, ...]
        all_labels: 所有类别列表

    返回:
        category_acc: {label: accuracy}
        confusion_pairs: [((true, pred), count), ...] 按 count 降序
    """
    # ↓↓↓ 你的代码 ↓↓↓
    category_correct = defaultdict(int)
    category_total = defaultdict(int)
    for r in results:
        category_total[r["true"]] += 1
        if r["true"] == r["pred"]:
            category_correct[r["true"]] += 1

    category_acc = {l: category_correct[l] / max(category_total[l], 1) * 100 for l in all_labels}

    confusion = Counter()
    for r in results:
        if r["true"] != r["pred"]:
            confusion[(r["true"], r["pred"])] += 1

    confusion_pairs = confusion.most_common()
    # ↑↑↑ 你的代码 ↑↑↑

    return category_acc, confusion_pairs

# 步骤 3：对比分析
base_acc_dict, base_conf = analyze_errors(base_results, LABELS)
sft_acc_dict, sft_conf = analyze_errors(sft_results, LABELS)

print(f"\n{'类别':<10} {'Base正确率':>10} {'SFT正确率':>10} {'提升':>8}")
print("-" * 42)
for label in LABELS:
    b = base_acc_dict.get(label, 0)
    s = sft_acc_dict.get(label, 0)
    print(f"{label:<10} {b:>9.1f}% {s:>9.1f}% {s-b:>+7.1f}%")

print(f"\nBase 最常见混淆对: {base_conf[0] if base_conf else 'N/A'}")
print(f"SFT 最常见混淆对: {sft_conf[0] if sft_conf else 'N/A'}")

improvements = {l: sft_acc_dict.get(l, 0) - base_acc_dict.get(l, 0) for l in LABELS}
best_improved = max(improvements, key=improvements.get)
print(f"\n结论: SFT 最大改善了「{best_improved}」类错误")

# --------- 验证 ---------
assert base_acc_dict is not None, "analyze_errors 返回了 None"
print("\n✅ 练习 4 完成！")

<details><summary>提示 1：按类别统计</summary>

```python
category_correct = defaultdict(int)
category_total = defaultdict(int)
for r in results:
    category_total[r["true"]] += 1
    if r["true"] == r["pred"]:
        category_correct[r["true"]] += 1
```
</details>

<details><summary>提示 2：混淆对统计</summary>

```python
confusion = Counter()
for r in results:
    if r["true"] != r["pred"]:
        confusion[(r["true"], r["pred"])] += 1
confusion_pairs = confusion.most_common()
```
</details>

<details><summary>提示 3：计算准确率</summary>

```python
category_acc = {l: category_correct[l] / max(category_total[l], 1) * 100 for l in all_labels}
return category_acc, confusion_pairs
```
</details>

---
## 练习 5：调超参（lr, epochs）观察效果（2:35-2:50）⭐⭐ 进阶

**目标**：理解学习率（learning rate）和训练轮数（epochs）对 SFT 效果的影响。

**任务**：
1. 用不同的 lr（如 1e-5, 5e-5, 2e-4）各训练一个模型
2. 记录最终 val loss
3. 画柱状图对比，找到最佳 lr

**企业启示**：超参调优（Hyperparameter Tuning）是 SFT 落地的必经之路。

**预期输出**：
```
一张柱状图，显示不同学习率对应的 val loss
结论: 最佳 lr = x.xe-x (val_loss = x.xxxx)
```

In [ ]:
# ====== 练习 5：调超参观察效果 ======

QUICK_STEPS = 50

lr_candidates = [1e-5, 5e-5, 2e-4]
results = {}

for test_lr in lr_candidates:
    print(f"\n--- 测试 lr={test_lr} ---")

    # ↓↓↓ 你的代码（创建模型、优化器、训练 QUICK_STEPS 步） ↓↓↓
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        test_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    test_model.config.pad_token_id = tokenizer.pad_token_id
    test_opt = torch.optim.AdamW(test_model.parameters(), lr=test_lr, weight_decay=0.01)

    step_count = 0
    test_model.train()
    for b in train_loader_sft:
        if step_count >= QUICK_STEPS:
            break
        ids = b["input_ids"].to(device)
        m = b["attention_mask"].to(device)
        y = b["labels"].to(device)
        out = test_model(input_ids=ids, attention_mask=m, labels=y)
        loss = out.loss
        test_opt.zero_grad()
        loss.backward()
        test_opt.step()
        step_count += 1
    # ↑↑↑ 你的代码 ↑↑↑

    # 计算验证集 loss（不要修改）
    val_loss = eval_loss_sft(test_model, val_loader_sft)
    results[test_lr] = val_loss
    print(f"  lr={test_lr} -> val_loss={val_loss:.4f}")
    del test_model, test_opt
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

# --------- 可视化（不要修改） ---------
plt.figure(figsize=(8, 4))
lrs = [str(lr) for lr in results]
vals = list(results.values())
colors = ['#10B981' if v == min(vals) else '#3B82F6' for v in vals]
plt.bar(lrs, vals, color=colors, edgecolor='black')
for i, (lr, v) in enumerate(zip(lrs, vals)):
    plt.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=11)
plt.xlabel('Learning Rate')
plt.ylabel('Val Loss')
plt.title('超参实验：不同学习率对 Val Loss 的影响')
plt.grid(axis='y', alpha=0.3)
plt.show()

best_lr = min(results, key=results.get)

# --------- 验证 ---------
assert len(results) == 3, f"应测试 3 个学习率，实际测试了 {len(results)} 个"
print(f"\n✅ 练习 5 完成！")
print(f"结论: 最佳 lr = {best_lr} (val_loss = {results[best_lr]:.4f})")

<details><summary>提示 1：准备模型和优化器</summary>

```python
test_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
test_model.config.pad_token_id = tokenizer.pad_token_id
test_opt = torch.optim.AdamW(test_model.parameters(), lr=test_lr, weight_decay=0.01)
```
</details>

<details><summary>提示 2：训练循环</summary>

```python
ids = b["input_ids"].to(device)
m = b["attention_mask"].to(device)
y = b["labels"].to(device)
out = test_model(input_ids=ids, attention_mask=m, labels=y)
loss = out.loss
test_opt.zero_grad()
loss.backward()
test_opt.step()
```
</details>

---
## 🎯 Mini-Project：SFT部署决策备忘录（15分钟）⭐⭐ 进阶

综合今天所学，写一份给技术经理的**部署建议**。

**任务**：汇总数据，回答"应该部署SFT模型还是继续用base+prompt？"

**提供模板结构，学员填数字和结论。**

In [ ]:
# ====== Mini-Project：SFT 部署决策备忘录 ======

# 用前面练习中的实际数字填充模板
base_acc_value = sum(1 for r in base_results if r['true']==r['pred'])/len(base_results)*100
sft_acc_value = sum(1 for r in sft_results if r['true']==r['pred'])/len(sft_results)*100
improvement = sft_acc_value - base_acc_value

# 找出 SFT 改善最大和仍然较弱的类别（来自练习 4 的 analyze_errors）
improvements_dict = {l: sft_acc_dict.get(l, 0) - base_acc_dict.get(l, 0) for l in LABELS}
best_improved_label = max(improvements_dict, key=improvements_dict.get)
worst_sft_label = min(sft_acc_dict, key=sft_acc_dict.get)
top_confusion = f"{sft_conf[0][0][0]} -> {sft_conf[0][0][1]} ({sft_conf[0][1]}次)" if sft_conf else "无明显混淆"

# 模型大小
model_size_mb = sum(p.numel() for p in sft_model.parameters()) * 4 / 1024 / 1024

# 显存估计
gpu_mem_gb = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0

# ↓↓↓ 你的代码（阅读模板，填写第 5、6 部分的结论） ↓↓↓
deployment_memo = f"""
{'='*60}
SFT 部署决策备忘录
{'='*60}

1. 准确率对比
   - Base 模型准确率:  {base_acc_value:.1f}%
   - SFT 模型准确率:   {sft_acc_value:.1f}%
   - 提升幅度:          {improvement:+.1f}%

2. 错误分析（来自练习 4）
   - SFT 改善最大的类别: {best_improved_label}
   - SFT 仍然较弱的类别: {worst_sft_label}
   - 最常见的混淆模式:    {top_confusion}

3. 训练成本估算
   - 训练数据量:     2400 条
   - 训练时间:       约 {sft_total_time/60:.1f} 分钟
   - 显存占用:       约 {gpu_mem_gb:.1f} GB

4. 部署考虑
   - 推理速度: 与 Base 模型基本相同
   - 模型大小: {model_size_mb:.0f} MB
   - 最低硬件: GPU 4GB+ 或 CPU（较慢）

5. 建议（二选一并说明理由）
   [x] 部署 SFT 模型，因为准确率提升 {improvement:+.1f}%，值得部署
   [ ] 继续用 Base+Prompt，因为: N/A

6. 下一步优化方向
   - 增加{worst_sft_label}类别的训练样本
   - 尝试 LoRA 等参数高效微调方法
{'='*60}
"""
# ↑↑↑ 你的代码 ↑↑↑

print(deployment_memo)
print(f"✅ 备忘录已生成！Base: {base_acc_value:.1f}%, SFT: {sft_acc_value:.1f}%")

---
## 本节总结

### 你学到了什么

| 知识点 | 关键收获 |
|:---|:---|
| LLM 生命周期 | Pretrain → SFT → Align → Deploy，企业通常从 SFT 切入 |
| 预训练 | 在大量文本上做 Next-Token Prediction，模型学会语言能力 |
| 数据质量 | 低质量数据 → 低质量模型，数据清洗至关重要 |
| 过拟合诊断 | 看 Train/Val Loss 曲线：val loss 上升 = 过拟合 |
| SFT | 用 instruction-response 数据教模型遵循指令 |
| Loss Masking | 只在 assistant 回复部分计算 loss，-100 屏蔽其他部分 |
| 超参调优 | lr 和 epochs 对效果影响大，需要实验确定 |

### 下一步预告

下午我们将学习 **LoRA 高效微调**——用不到 1% 的参数实现接近全量微调的效果。